# Cross-Basin Generalisation for Tropical Cyclone Forecasting using TropiCycloneNet

**Team:** Loïc Bouxirot, Yasmin Akhmedova, Samuel Zhang

**Module:** ELEC70127 - ML for Tackling Climate Change

**Date:** March 2026

## Introduction

Tropical cyclone (TC) forecasting in data-scarce ocean basins is limited by small training
sets. Traditional forecasting systems used by meteorological agencies require extensive
historical records that simply do not exist in many regions [1]. Transfer learning from
data-rich to data-scarce basins is a possible solution, and there is evidence this works:
a Western North Pacific-trained transformer fine-tuned on the Eastern North Pacific
outperformed established forecasting systems [7]. This project investigates whether deep
learning models trained on the **Western Pacific (WP)**, one of the most active and well-
observed basins, can generalise to the **South Pacific (SP)**, a data-scarce basin in the
opposite hemisphere.

The WP to SP transfer is particularly challenging because the hemisphere flip introduces a
**Coriolis reversal**: WP storms predominantly move towards the northwest and west (NW/W),
while SP storms move towards the southeast, west, and southwest (SE/W/SW) [1]. A naive WP-
trained model will systematically mispredict SP storm directions. Multi-basin models
achieve 61-73% better track prediction than WP-only models on Southern Hemisphere basins
[1]. However, the underlying atmospheric physics, specifically sea surface temperature
(SST) and vertical wind shear, is shared across both basins [2]. Our exploratory analysis
confirms this: as Figure 1 shows, WP and SP have substantially overlapping SST and wind
shear distributions. **This suggests that physics-informed features should transfer even
if direction labels do not.**

_Figure 1: SST and vertical wind shear distributions across all six TropiCycloneNet Dataset (TCND) basins. WP (brown) and SP (orange) overlap substantially on both variables, supporting the hypothesis that intensity-related physics is shared across hemispheres._

![SST and wind shear distributions by basin](figures/sst_shear_by_basin.png)

### What we explore

We investigate two fundamentally different approaches to cross-basin transfer, each
targeting a different aspect of the prediction problem:

| Core model | Family | What it captures | How we got there |
| --- | --- | --- | --- |
| **U-Net + FiLM** | Spatial (encoder-decoder) | Local wind patterns around the storm → strong at **direction** prediction | - **SE (Squeeze-and-Excitation) attention** at every encoder/decoder level: learns which input variables (e.g. wind, SST, geopotential) matter most and adjusts their influence accordingly <br> - **Augmentation suite** to combat overfitting on ~1,816 samples: Mixup (blends two samples into one), CutOut (masks random patches of the input grid), Gaussian noise (adds small random perturbations), and channel dropout (randomly zeroes out entire input variables) <br> - **FiLM temporal conditioning** [3]: tells the model *when* in a storm's lifecycle each sample occurs (storm progress, hour, season), which improved intensity prediction |
| **U-FNO** | Spectral-spatial hybrid | Global atmospheric patterns (SST, shear) → strong at **intensity** prediction | - Started with a baseline **FNO** [4] that learns in the frequency domain via a Fast Fourier Transform (FFT). Instead of scanning small local patches like a regular convolution, the FFT processes the entire grid at once, capturing large-scale atmospheric patterns. However, it overfitted severely on our small dataset <br> - Improved to **FNO v2** with reflect padding (mirrors the grid edges outward so the FFT doesn't see artificial discontinuities at the boundaries) and FiLM conditioning <br> - Combined spectral and spatial into **U-FNO** [5]: each layer runs three parallel branches (spectral, U-Net, residual) and merges them using **learnable gate weights**, which are trainable parameters that control how much each branch contributes, so the model automatically discovers the best mix of global and local features |

### Model naming convention

Throughout this report and the supplementary notebooks, we use a systematic naming scheme.
**Family 1** covers spatial (encoder-decoder) architectures; **Family 2** covers spectral
(Fourier) architectures. Letters indicate iterations within each family:

| ID | Short Name | Family | Description |
| --- | --- | --- | --- |
| **1a** | U-Net | Spatial | Baseline encoder-decoder with SE attention and DropPath |
| **1b** | U-Net + FiLM | Spatial | Added temporal conditioning via FiLM (final spatial model) |
| **2a** | FNO | Spectral | Baseline frequency-domain model (2 spectral layers) |
| **2b** | FNO + FiLM | Spectral | Added reflect padding + FiLM temporal conditioning |
| **2c** | U-FNO | Spectral | Hybrid spectral-spatial with learnable gating (final spectral model) |

Models **1b** and **2c** are the two final models carried through the full evaluation
pipeline and ablation analysis. The intermediate models (1a, 2a, 2b) document the design
progression and are compared in the supplementary model comparison notebook.

### Prediction task

Each model performs **dual-head classification** at every 6-hourly timestep, predicting
what the storm will do over the **next 24 hours**:

- **Direction (8 classes):** E, SE, S, SW, W, NW, N, NE, representing the compass
  direction the storm will move. The full 360° of possible movement is divided into 8
  equal sectors of 45° each, which is the classification scheme used in TCND [1]. At this
  resolution, each class covers a meaningful range of motion. Using more classes (e.g. 16)
  would make distinctions that are hard to predict 24 hours ahead, while fewer classes
  (e.g. 4) would lose operationally useful detail.

- **Intensity change (4 classes):** Weakening, Steady, Slow-intensification, Rapid-
  intensification, representing how the storm's maximum wind speed will change. These
  categories are pre-defined in TCND [1] and align with operational forecasting practice:
  the distinction between slow and rapid intensification is particularly important because
  rapid intensification events are the hardest to forecast and most dangerous [2].

**Why 24 hours?** This is the standard operational forecast window used by meteorological
agencies worldwide (e.g. NHC, RSMC Tokyo). Both labels (`future_direction24`,
`future_inte_change24`) are pre-computed in the TCND dataset for this horizon. The
6-hourly observation interval means we predict 4 timesteps ahead.

Our three-stage evaluation pipeline consists of:
1. **Zero-shot transfer**: train on WP, apply hemisphere-aware preprocessing, evaluate on
   SP with no fine-tuning.
2. **Fine-tuning**: fine-tune on a small SP training set (12 storms, ~354 samples) and re-
   evaluate.
3. **Transfer gap analysis**: quantify the performance drop from in-basin to cross-basin
   and how much fine-tuning recovers.

## Part 1: Data & Preprocessing

We use the **TropiCycloneNet Dataset (TCND)** [1], a multimodal tropical cyclone dataset
covering 1950–2023 across six ocean basins. Our preprocessing pipeline (`supplementary-notebooks/preprocessing/data-preprocessing-pipeline.ipynb`) transforms raw TCND data into
model-ready tensors.

### 1.1 Dataset Overview

Three types of data are available per 6-hourly timestep:

_Table 1: Data Types in TCND_

| # | Data type | Format | Content |
| --- | --- | --- | --- |
| 1 | **Data3D** (gridded atmospheric fields) | NetCDF, a standard scientific file format for storing multidimensional weather/climate data | - 81×81 pixel grid centred on the storm (20°×20° at 0.25° resolution) <br> - SST (sea surface temperature) <br> - u/v wind (horizontal/vertical) at 4 pressure levels (200, 500, 850, 925 hPa) <br> - Geopotential height at 4 pressure levels <br> - 13 variables total |
| 2 | **Env-Data** (environmental context) | NumPy arrays, a standard Python numerical format | - 94 pre-computed environmental variables per timestep <br> - Includes the 24h direction and intensity-change labels we use as targets |
| 3 | **Data1D** (storm track history) | Tab-separated text files, one row per timestep | - Per-storm track files with latitude/longitude offsets (displacement from the storm centre), normalised wind speed and pressure |

### WP vs SP Comparison

We train on WP and test on SP. The table below highlights two things: (1) WP is much
larger than SP, giving us a data-rich source domain, and (2) the storm directions differ
between hemispheres but the underlying physics is similar, which is why cross-basin
transfer is worth attempting.

_Table 2: Basin Comparison_

|  | WP (train) | SP (test) | What this tells us |
| --- | --- | --- | --- |
| Storms | 87 | 30 | WP has 2.9x more storms, making it a strong source domain for transfer learning |
| Timesteps | 2,699 | 922 | SP is data-scarce, which is exactly the problem we are trying to solve |
| Top directions | NW, W | SE, W | WP storms mostly head northwest while SP storms mostly head southeast, because the Coriolis effect steers storms in opposite directions in each hemisphere. This is why a WP-trained model will mispredict SP directions without correction. |

### 1.2 Preprocessing Pipeline

The preprocessing pipeline applies four key transformations. Full implementation is in
`supplementary-notebooks/preprocessing/data-preprocessing-pipeline.ipynb`.

**1. Basin-leaking feature removal**
*Why:* Some features in the raw data directly reveal which ocean basin a storm belongs to.
If we leave these in, the model can take shortcuts (e.g. "this is WP, so predict NW")
instead of learning generalisable physics. We need to force the model to rely on
atmospheric patterns that work across basins.
*What:* 54 of the 94 environmental dimensions encode basin identity (area one-hot,
longitude/latitude bins, raw coordinates). We strip these, retaining only the 40 physics-
relevant dimensions: wind, move velocity, intensity class, month, and historical
direction/intensity labels.

**2. Hemisphere-aware transforms (SP only)**
*Why:* WP is in the Northern Hemisphere, SP is in the Southern Hemisphere. Storms rotate
in opposite directions due to the Coriolis effect, so a "northwestward" storm in WP is
physically analogous to a "southwestward" storm in SP. Without correcting for this, the
model would see completely different direction patterns and fail to transfer.
*What:* Direction labels are reflected along the north/south axis so that equivalent storm
movements in each hemisphere map to the same class (e.g. NW in WP and SW in SP both become
"away from the equator and westward"). Meridional wind (v) is also sign-flipped so that
"towards the pole" has a consistent sign in both hemispheres.

**3. Derived physics channels**
*Why:* The raw grid data contains wind speed and geopotential height at different
altitudes, but two quantities known to be critical for cyclone behaviour are not given
directly:
- **Vertical wind shear** (the difference between upper-atmosphere and lower-atmosphere
  winds) tears apart a cyclone's structure. High shear inhibits intensification, while low
  shear allows it [2].
- **Relative vorticity** (how strongly the air is spinning at low altitude) measures the
  cyclone's rotational strength. Higher vorticity indicates a more organised, intensifying
  storm.

By computing these explicitly rather than expecting the model to learn them from raw wind
fields, we give it direct access to known physical drivers of storm intensity.

*What:* Two variables are appended to the 13 base grid variables, giving **15 total**:
- **Wind shear magnitude** = $\sqrt{(u_{200} - u_{850})^2 + (v_{200} - v_{850})^2}$, i.e.
  how different the wind is at 200 hPa (upper atmosphere, ~12 km altitude) vs 850 hPa
  (lower atmosphere, ~1.5 km altitude). Large values mean the upper and lower winds are
  pulling the storm in different directions.
- **850 hPa relative vorticity** = $\frac{\partial v_{850}}{\partial x} - \frac{\partial
  u_{850}}{\partial y}$ (centred finite differences), i.e. how much the low-level wind
  field is rotating around the storm centre. Computed from spatial gradients of the u and
  v wind components.

**4. Normalisation**
*Why:* Different atmospheric variables have very different scales (e.g. SST in Kelvin
~300, wind in m/s ~10, geopotential in m²/s² ~50,000). Without normalisation, the model
would be dominated by whichever variable has the largest numbers. We also compute
statistics only from WP training data to avoid leaking any information about SP into the
model before evaluation.
*What:* Per-channel z-score computed exclusively from WP training data and applied to all
splits.

**Missing Data1D handling:** If a timestep has no matching Data1D entry, the entire
timestep is dropped from the dataset (grid, env, and labels are all excluded). We do not
zero-fill because zero wind speed or zero pressure has a real physical meaning and would
feed the model false information. Drop counts are logged per storm and per split.

### 1.3 Data Reduction & Final Split Sizes

Not every timestep in the raw dataset can be used for training. Two filtering stages
reduce the data: (1) some WP storms have no matching track data in the subset we use, and
(2) timesteps too close to the end of a storm lack a valid 24h-ahead label. The table
below shows how many timesteps we lose at each stage:

_Table 3a: Data Reduction Summary_

| Stage | Storms | Timesteps | What happens |
| --- | --- | --- | --- |
| **Raw TCND** | 161 | 5,484 | All available 6-hourly observations across WP (131 storms, 4,562 timesteps) and SP (30 storms, 922 timesteps) |
| **After Data1D filtering** | 117 | 3,621 | 44 WP storms (1,863 timesteps) dropped. The TCND dataset stores gridded fields (Data3D) and environmental features (Env-Data) for these storms, but the corresponding Data1D track files (which provide latitude/longitude offsets, wind, and pressure per timestep) are not included in the TCND test subset we downloaded. Since our models require all three modalities as input, storms without Data1D entries cannot be used. The missing storms are predominantly from 2020–2021. All 30 SP storms are unaffected. |
| **Valid 24h labels** | 117 | 3,153 (57%) | 468 timesteps are dropped because they are too close to the end of their storm to have a valid 24h-ahead label |

The remaining 3,153 labelled timesteps are then split across five subsets for our training
and evaluation pipeline:

_Table 3b: Final Split Sizes_

| Split | Storms | Timesteps | Valid Labels | Purpose |
| --- | --- | --- | --- | --- |
| `wp_train` | 69 | 2,092 | 1,816 | Train models on WP data |
| `wp_val` | 18 | 607 | 535 | Tune hyperparameters and early stopping on WP |
| `sp_test` | 15 | 427 | 367 | Test how well WP-trained models work on SP (no SP data seen during training) |
| `sp_ft_train` | 12 | 402 | 354 | Fine-tune WP-trained models on a small amount of SP data |
| `sp_ft_val` | 3 | 93 | 81 | Validate fine-tuning (only 3 storms, so very limited) |
| **Total** | **117** | **3,621** | **3,153** | |

**Key takeaway:** With only ~1,816 usable training samples, our dataset is small by deep
learning standards. This means models with too many parameters (like ResNet-152 at 58.7M)
will memorise the training data rather than learn generalisable patterns. Throughout our
experiments, smaller models with strong regularisation (augmentation, dropout, early
stopping) consistently outperformed larger ones.

### 1.5 Temporal Feature Extraction

The three data types described in Section 1.1 (gridded fields, environmental variables,
and storm track history) capture *what* is happening around the storm at a given moment,
but none of them tell the model *when* that moment occurs. Knowing the time context
matters because cyclone behaviour changes systematically throughout a storm's lifecycle
(early storms tend to intensify, late-stage storms tend to weaken) and across seasons
(peak season storms are generally more intense). To give the model access to this
information, we compute a 6-dimensional temporal vector per timestep.

_Table 4: Temporal Features_

| Index | Feature | Formula | What it captures |
|-------|---------|---------|-----------------|
| 0 | `storm_progress` | $i / (N_t - 1)$ | How far through its lifecycle the storm is (0 = just formed, 1 = about to dissipate). Helps the model learn that early-stage and late-stage storms behave differently. |
| 1 | `hour_sin` | $\sin(2\pi \cdot h / 24)$ | Time of day encoded as a position on a circle (sin component). With 6-hourly data, this produces 4 distinct values for 00, 06, 12, 18 UTC. |
| 2 | `hour_cos` | $\cos(2\pi \cdot h / 24)$ | Time of day (cos component). Both sin and cos are needed because sin alone cannot distinguish e.g. 06 UTC from 18 UTC. |
| 3 | `month_sin` | $\sin(2\pi \cdot (m-1) / 12)$ | Season encoded as a position on a circle (sin component). Captures that storms forming in peak season (Jul-Oct for WP) tend to be more intense. |
| 4 | `month_cos` | $\cos(2\pi \cdot (m-1) / 12)$ | Season (cos component). Together with month_sin, ensures December and January are treated as adjacent rather than far apart. |
| 5 | `storm_duration_norm` | $N_t / N_{\max}$ | How long-lived this storm is relative to the longest storm in the dataset. Longer storms tend to be more intense, so this gives the model a structural signal about storm strength. |

**Design choices:**

- **Cyclical encoding** (features 1-4): Raw hour or month values would imply that hour 23
  and hour 0, or December and January, are far apart. Sin/cos encoding places them
  adjacent on a unit circle. Both components are needed because sin alone is ambiguous
  ($\sin$ is the same for hour 6 and hour 18).
- **Progress vs duration** (features 0 and 5): These capture complementary information.
  `storm_progress` varies *within* a storm (where are we in the lifecycle?), while
  `storm_duration_norm` is constant *within* a storm but varies *across* storms (how long-
  lived is this storm overall?). Longer storms tend to be more intense, so duration gives
  the model a useful structural signal.

These 6 temporal features are not fed directly into the model as extra inputs. Instead,
they are injected via **FiLM conditioning** [3] (described fully in Part 2): the temporal
vector is passed through a small neural network that produces per-layer scale and shift
values, which modify how each convolutional block processes the spatial data. This lets
the model adjust its behaviour depending on when in the storm's lifecycle the current
observation falls.

_Full implementation: `supplementary-notebooks/preprocessing/temporal-feature-extraction.ipynb`_

## Part 2: Spatial Family — U-Net (Models 1a–1b)

We started with a standard **U-Net (Model 1a)** encoder-decoder as our spatial baseline.
It performed reasonably well on direction prediction (55.9% WP direction accuracy), but
its intensity accuracy was limited (54.2%). After adding **FiLM temporal conditioning
(Model 1b)**, which tells the model *where* in a storm's lifecycle each sample occurs,
intensity improved substantially (+4.3pp) while direction also gained slightly. The FiLM
variant became our final U-Net model, so we focus on it here while briefly describing the
baseline it builds on.

_Full implementations: `supplementary-notebooks/models/model-1a-unet.ipynb` (Base U-Net), `supplementary-notebooks/models/model-1b-unet-film.ipynb` (U-Net + FiLM)_

### 2.1 Architecture

#### Base U-Net

The encoder-decoder follows a standard U-Net structure:

- **Encoder:** 4 levels (32→64→128→256 channels) with a 512-channel bottleneck. Each level
  has residual blocks with skip connections to the decoder.
- **SE channel attention** at every level  -  learns which of the 15 input channels matter
  most (e.g. upweighting SST and wind, downweighting less informative geopotential
  channels).
- **Decoder:** mirrors the encoder, using skip connections to recover spatial detail.
- **Classification heads:** the decoder output is globally averaged to a 32-d vector,
  concatenated with environmental (40-d) and 1D track (4-d) features, and fed to two
  separate heads (direction and intensity).

Key design choices:

- **Aggressive augmentation suite** to combat overfitting on only ~1,816 training samples
  -  Mixup (α=0.2), CutOut (2×16×16 patches), Gaussian noise (σ=0.05), channel dropout
  (p=0.15), and EMA weight averaging. We deliberately avoid spatial flips, since flipping
  a storm image would reverse the physical direction of movement, corrupting the labels.

- **Small base channel width (32)**  -  HPO via Optuna (30 trials × 30 epochs) showed the
  model is **data-limited, not capacity-limited**: base_channels=32 outperformed 64,
  confirming that more parameters don't help when training data is scarce.

#### U-Net + FiLM

The base U-Net sees each timestep in isolation  -  it knows *what* the atmosphere looks
like right now, but not *when* in the storm's lifecycle that snapshot was taken. To give
the model temporal awareness, we add **Feature-wise Linear Modulation (FiLM)** [3]:

- **Temporal conditioning signal:** a small MLP takes the 6-d temporal vector from Section
  1.5 (storm progress, time of day, season, storm duration) and produces a per-channel
  scale (γ) and shift (β).

- **Where it's applied:** after the second BatchNorm in every encoder and decoder block,
  each channel is transformed as $y = \gamma x + \beta$. This lets the model adjust how it
  weighs different atmospheric features depending on the storm's temporal context  -  for
  example, SST might matter more during early intensification than during a storm's decay
  phase.

- **Identity initialisation:** γ=1 and β=0 at the start of training, so the model begins
  *exactly equivalent* to the base U-Net. Temporal conditioning has to *earn* its
  influence by learning useful γ and β values during training. If the temporal features
  turn out not to be helpful, the model gracefully falls back to the baseline rather than
  being harmed by a poorly-initialised conditioning pathway.

All hyperparameters are identical to the baseline: `BASE_CHANNELS=32, N_LEVELS=4,
EPOCHS=300, PATIENCE=50`.

_Checkpoints: `unet_film_best_wp.pt` (trained on WP only  -  used for zero-shot SP evaluation), `unet_film_best_ft.pt` (fine-tuned on SP  -  used for fine-tuned SP evaluation)_

### 2.2 What FiLM changed

Adding temporal conditioning had a clear, asymmetric effect on the two tasks:

_Table 5: U-Net baseline vs U-Net + FiLM (WP Validation)_

| Metric | U-Net baseline | U-Net + FiLM | Difference |
| --- | --- | --- | --- |
| Direction Accuracy | 0.559 | **0.563** | +0.4pp |
| Direction F1 | 0.341 | **0.371** | +3.0pp |
| Intensity Accuracy | 0.542 | **0.585** | **+4.3pp** |
| Intensity F1 | 0.448 | **0.463** | **+1.5pp** |
| Parameters | 9.8M | 10.0M | +0.2M |
| Epochs (early stop) | 155/300 | 189/300 | Similar |

**Why the difference?** Temporal features (storm progress, hour/month cycles, storm
duration) encode information about *when* intensification is likely, e.g. storms that form
in peak season and are in mid-lifecycle tend to intensify more. This directly helps
intensity prediction. Direction, on the other hand, is mainly determined by the spatial
wind fields in the grid, which the model already has access to. Knowing the time of day
adds little.

## Part 3: Spectral Family — FNO to U-FNO (Models 2a–2c)

The U-Net + FiLM works well for direction, but intensity depends on large-scale
thermodynamic conditions (how warm is the ocean? how much wind shear is there?) that local
convolutions struggle to capture. To address this, we explored the **Fourier Neural
Operator (FNO)** [4], which learns in the frequency domain rather than scanning local
patches. After iterating through two earlier versions, we arrived at the **U-FNO** [5], a
hybrid that combines spectral and spatial branches. It achieved 60.0% WP intensity
accuracy (vs 58.5% for U-Net + FiLM), confirming that spectral methods better capture the
global physics driving intensification.

We iterated through three versions before arriving at our final model:

- **Model 2a — FNO baseline** (2 spectral layers, 80 hidden channels): Overfitted
  severely. The FFT assumes periodic boundary conditions, which creates artifacts at the
  edges of our 81×81 grid.
- **Model 2b — FNO v2** (3 spectral layers, reflect padding, FiLM): Padding fixed the
  boundary problem and FiLM added temporal awareness. Achieved competitive intensity
  accuracy, but direction remained weak.
- **Model 2c — U-FNO** (hybrid): Combines spectral convolutions with a lightweight U-Net
  branch, letting the model use *both* global and local features. This is our final FNO-
  family model, and we focus on it here while briefly describing the building blocks it
  inherits from the earlier versions.

_Full implementations: `supplementary-notebooks/models/model-2a-fno.ipynb` (FNO baseline), `supplementary-notebooks/models/model-2b-fno-film.ipynb` (FNO v2), `supplementary-notebooks/models/model-2c-ufno.ipynb` (U-FNO)_

### 3.1 What we learned from the earlier FNO versions

Each iteration solved a specific problem, and the U-FNO incorporates the solutions from
both:

- **From the FNO baseline: spectral convolution.** Instead of sliding a small filter over
  the image (like a regular convolution), the FNO transforms the entire grid into the
  frequency domain using an FFT, multiplies by learnable weights for the lowest Fourier
  modes, and transforms back. This lets the model learn *global* atmospheric patterns
  (e.g. basin-wide SST gradients) rather than local features. However, the baseline
  overfitted quickly and produced boundary artifacts because the FFT assumes the grid
  wraps around.

- **From FNO v2: reflect padding + FiLM.** Two fixes that we carried forward into U-FNO:
  (1) **reflect padding**, which extends the grid by 9 pixels on each side before the FFT
  and crops afterward, eliminating the boundary artifacts; (2) **FiLM conditioning**, the
  same temporal modulation as in Part 2 (Section 2.1), injected after BatchNorm in each
  spectral block to give the model lifecycle and seasonal awareness.

### 3.2 U-FNO Architecture

The U-FNO [5] runs **three parallel branches** through each layer and blends them with
learnable gate weights:

- **Spectral branch** (`PaddedSpectralConv2d`): global frequency-domain features with
  reflect padding. Captures basin-wide patterns like SST gradients and large-scale shear.
- **U-Net branch** (`UNetBranch`): local spatial features via a lightweight single-level
  encode-decode. Captures local wind structure around the storm centre.
- **Residual branch** (`nn.Conv2d(1×1)`): identity/skip pathway. Preserves input features
  and stabilises training.

The three branch outputs are combined as: `output = g₀·spectral + g₁·unet + g₂·residual`,
where the gate weights `g` are learned via softmax (initialised at ⅓ each). After gating,
the output goes through BatchNorm → FiLM → GELU → dropout, plus a residual connection from
the input.

**Configuration:** `HIDDEN_CH=32, N_MODES=12, N_LAYERS=3, PADDING=9, TIME_EMB_DIM=64`,
only **1.0M parameters** (vs 10.0M for U-Net+FiLM).

_Checkpoints: `ufno_best_wp.pt` (trained on WP only), `ufno_best_ft.pt` (fine-tuned on SP)_

### 3.3 What the learned gates tell us

A key advantage of the U-FNO's gated architecture is that it is *interpretable*: because
the three branches are combined via learned softmax weights, we can inspect those weights
after training to see which type of feature the model found most useful. This is analogous
to Section 2.2 where we compared the U-Net with and without FiLM to see if temporal
conditioning helps - here, we can directly read off how much the model relies on global
(spectral) vs local (U-Net) vs identity (residual) features.

_Table 5c: Learned Gate Weights_

| Layer | Spectral | U-Net | Residual |
| ---: | ---: | ---: | ---: |
| 1 | **0.40** | 0.27 | 0.33 |
| 2 | **0.40** | 0.30 | 0.30 |
| 3 | **0.38** | 0.31 | 0.31 |

Three takeaways:

- **Global patterns dominate:** the spectral branch receives the highest weight in every
  layer (38-40%), confirming that large-scale atmospheric features (SST gradients, basin-
  wide shear) are the most informative signal for cyclone forecasting.
- **Local features grow with depth:** the U-Net branch weight increases from 27% in layer
  1 to 31% in layer 3, suggesting that the model first captures broad structure and then
  refines with local spatial detail in later layers.
- **All branches contribute:** the near-uniform distribution means no branch is redundant.
  If we had found, say, 90% spectral and 5% U-Net, we could have dropped the U-Net branch
  entirely. Instead, the model genuinely benefits from combining both perspectives.

## Part 4: Results & Analysis

We evaluate our two final models, **U-Net+FiLM (Model 1b)** and **U-FNO (Model 2c)**,
across three stages: in-basin training (WP), zero-shot cross-basin transfer (SP), and
fine-tuning on a small SP subset. The headline finding is that **these two models excel at
different tasks**: U-Net+FiLM is best at predicting storm direction, while U-FNO is best
at predicting intensity change.

_Full comparison: `supplementary-notebooks/supplementary-analysis/model-comparison.ipynb`_

### 4.1 In-Basin Performance (WP Validation)

_Table 6: WP Validation Results_

| Metric | U-Net + FiLM | U-FNO |
| --- | --- | --- |
| **Direction Accuracy** | **0.563** | 0.479 |
| Direction F1 (macro) | **0.371** | 0.380 |
| **Intensity Accuracy** | 0.585 | **0.600** |
| Intensity F1 (macro) | 0.463 | **0.501** |
| Parameters | 10.0M | **1.0M** |
| Epochs (early stop) | 151/300 | 41/150 |

**Direction vs intensity: two different problems.** U-Net+FiLM leads direction by over 8pp
(56.3% vs 47.9%), while U-FNO leads intensity by 1.5pp (60.0% vs 58.5%). This makes
physical sense: direction is determined by *local* wind patterns around the storm (where
the U-Net's spatial convolutions excel), while intensity change is driven by *global*
conditions like SST and vertical wind shear (where the FNO's spectral convolutions excel).

U-FNO achieves this with **10× fewer parameters** (1.0M vs 10.0M), but overfits faster,
stopping at epoch 41 vs 151 for U-Net+FiLM. The spectral convolutions are highly
expressive but lack the inductive biases (locality, skip connections, augmentation) that
help the U-Net regularise on our small dataset.

### 4.2 Zero-Shot Cross-Basin Transfer (SP Test)

_Table 7: Zero-Shot SP Results_

| Metric | U-Net + FiLM | U-FNO |
| --- | --- | --- |
| **Direction Accuracy** | **0.338** | 0.322 |
| Direction F1 (macro) | 0.234 | **0.239** |
| **Intensity Accuracy** | 0.395 | **0.425** |
| Intensity F1 (macro) | 0.283 | **0.280** |

_Table 8: Transfer Gap (WP → SP)_

| Gap | U-Net + FiLM | U-FNO |
| --- | --- | --- |
| Direction | −22.5pp | −15.7pp |
| Intensity | −19.0pp | **−17.5pp** |

Both models suffer a large drop when applied to SP without any fine-tuning. This is
expected, given the hemisphere flip. U-Net+FiLM maintains a slight edge on direction
(33.8% vs 32.2%), while **U-FNO leads on intensity** (42.5% vs 39.5%).

Notably, U-FNO has a **smaller transfer gap** on both tasks (−15.7pp dir vs −22.5pp),
suggesting that spectral convolutions learn more basin-invariant representations than
purely spatial convolutions. The global Fourier modes capture thermodynamic patterns that
are shared across hemispheres.

### 4.3 Fine-Tuned Performance (SP Test)

After zero-shot evaluation, we fine-tune each model on a small SP training set (12 storms,
354 samples) and re-evaluate.

_Table 9: Fine-Tuned SP Results_

| Metric | U-Net + FiLM | U-FNO |
| --- | --- | --- |
| **Direction Accuracy** | **0.346** | 0.226 |
| Direction F1 (macro) | **0.249** | 0.187 |
| **Intensity Accuracy** | **0.458** | 0.441 |
| Intensity F1 (macro) | **0.382** | 0.379 |

_Table 10: Fine-Tuning Recovery (vs Zero-Shot)_

| Recovery | U-Net + FiLM | U-FNO |
| --- | --- | --- |
| Direction | **+0.8pp** | −9.6pp |
| Intensity | **+6.3pp** | +1.6pp |

Fine-tuning tells an interesting story:

- **Intensity improves for both models**, confirming that the underlying physics (SST,
  shear) transfers and just needs recalibration for SP conditions.
- **U-Net+FiLM recovers modestly on direction** (+0.8pp), showing that FiLM conditioning
  allows some adaptation to SP's different seasonal patterns.
- **U-FNO collapses on direction** (−9.6pp), suggesting that its compact 1.0M parameters
  are vulnerable to catastrophic forgetting when fine-tuned on only 354 samples. The
  spectral features that worked well zero-shot are destabilised by the small SP training
  set.
- After fine-tuning, **U-Net+FiLM leads both tasks**, reaching 45.8% intensity accuracy.

### 4.4 Overfitting Analysis

With only ~1,816 training samples, overfitting is the dominant challenge. The table below
compares how each model family handled it:

_Table 11: Overfitting Across Model Families_

| Model | Parameters | Params-per-Sample | Epochs Before Stop | Severity |
| --- | ---: | ---: | --- | --- |
| FNO (baseline) | 7.4M | 4,074:1 | 114/300 | Severe: large overfit gap (0.89) despite extended training |
| FNO v2 | 9.3M | 5,121:1 | 135/300 | Moderate: reflect padding and FiLM help, overfit gap 0.96 |
| **U-FNO** | **1.0M** | **551:1** | **41/150** | Moderate: compact gated branches reduce overfitting, overfit gap 1.03 |
| U-Net (baseline) | 9.8M | 5,396:1 | 134/300 | Controlled: augmentation suite keeps training productive, overfit gap 0.73 |
| **U-Net + FiLM** | **10.0M** | **5,507:1** | **151/300** | Controlled: FiLM adds minimal overhead, same augmentation, overfit gap 0.57 |

The key contrast: **U-Net tolerates high parameter counts** because its augmentation suite
(Mixup, CutOut, noise, channel dropout, EMA) acts as strong regularisation. **FNO without
augmentation overfits quickly** at similar parameter counts (7.4M), but the U-FNO's
compact 1.0M parameters and gated architecture mitigate this.

**Results:** By fixing the grid and 1D inputs to their means, we ensure that SHAP values
reflect the marginal contribution of each environmental feature without confounds from the
spatial data. The analysis uses 100 background samples and 200 explanation samples.

## Part 5: Explainability & Ablation

Parts 2–4 showed that U-Net+FiLM (Model 1b) is better at direction and U-FNO (Model 2c) is
better at intensity, but not *why*. Understanding which input features drive each
prediction is important both for trusting the models and for guiding future improvements.
We investigate this through two complementary approaches, applied to both final models
side by side:

- **Modality, channel, and feature group ablation** (Section 5.1): systematically zero out
  input modalities (grid, env, 1D, time), individual grid channel groups, and
  environmental feature groups, then compare the accuracy drop across both models. This
  reveals whether U-Net+FiLM and U-FNO rely on the same inputs or different ones.
- **SHAP comparison** (Section 5.2): gradient-based SHAP values on environmental features
  for both models, quantifying whether the per-feature importance profiles differ between
  architectures.

_Full analysis: `supplementary-notebooks/supplementary-analysis/ablation-and-shap-comparison.ipynb`_

### 5.1 Ablation Comparison

Ablation measures how much a model relies on a given input by **zeroing it out** and
measuring the accuracy drop. A large drop means the model depends heavily on that input;
no drop means it is redundant. We run the same ablation framework on both final models
(U-Net+FiLM and U-FNO), with an additional "No Time" configuration that zeros out the
temporal features to test whether FiLM conditioning matters at inference time.

We apply three increasingly fine-grained levels of ablation:

1. **Modality-level:** zero out an entire input stream (grid, env, 1D, or time). This
   answers the broadest question: *which type of data does each model actually need?* For
   instance, if removing the grid barely hurts U-FNO but destroys U-Net+FiLM, we know the
   two models extract fundamentally different signals.
2. **Grid channel group (leave-one-out):** zero out one group of grid channels at a time
   (e.g. all u_wind levels, or SST) while keeping everything else. The modality-level
   results tell us the grid matters, but not *which atmospheric variables within it*
   matter. This level identifies whether the models rely on the same channels (e.g. both
   need wind) or different ones (e.g. U-Net uses wind while U-FNO uses SST).
3. **Environmental feature group (leave-one-out):** same idea applied to the 40-d env
   vector - zero out one semantically grouped subset (e.g. intensity_class, month). Since
   the env vector contains pre-computed summary statistics rather than raw spatial fields,
   this tells us which high-level meteorological quantities complement what the model
   already extracts from the grid.

**Modality-level ablation**

We zero out entire input streams and measure the accuracy drop for each model:

_Table 12a: Modality Ablation (WP Validation)_

| Configuration | U-Net+FiLM Dir Drop | U-FNO Dir Drop | U-Net+FiLM Int Drop | U-FNO Int Drop |
| --- | ---: | ---: | ---: | ---: |
| **Baseline** | **56.4%** | **55.7%** | **57.4%** | **63.0%** |
| No Grid | +24.1pp | +22.8pp | +18.9pp | +23.7pp |
| No Env | +1.5pp | +0.2pp | +7.1pp | +3.9pp |
| No 1D | -2.2pp | +0.7pp | -2.6pp | +1.7pp |
| No Time | +0.7pp | -0.9pp | +5.2pp | +6.9pp |
| Only Grid | -1.9pp | +0.6pp | +5.2pp | +7.3pp |
| Only Env | +26.0pp | +22.4pp | +19.1pp | +32.0pp |
| Only 1D | +31.6pp | +22.4pp | +23.0pp | +29.2pp |

Four findings:

- **Grid data is essential for both models.** Removing the grid causes the largest single
  drop for both architectures (~24pp direction, ~19-24pp intensity). The grid alone ("Only
  Grid") recovers nearly all direction accuracy, confirming that spatial atmospheric
  fields are the dominant input.
- **Temporal features (FiLM) matter for intensity, not direction.** Zeroing out the time
  vector drops intensity by +5.2pp (U-Net+FiLM) and +6.9pp (U-FNO), but has negligible
  effect on direction. This confirms that storm lifecycle and seasonal context help
  predict intensification but not movement.
- **Env features help U-Net+FiLM intensity more than U-FNO.** Removing env costs
  U-Net+FiLM 7.1pp on intensity vs only 3.9pp for U-FNO. The spectral model extracts
  intensity-relevant information (SST, shear) directly from the grid, making it less
  reliant on the pre-computed environmental vector.
- **1D track data is redundant.** Removing it has no meaningful effect (or even slightly
  improves performance), confirming that the grid and env already contain the information
  1D provides.

**Grid channel group ablation (leave-one-out)**

_Table 12b: Grid Channel Group Ablation_

| Channel Group | U-Net+FiLM Dir Drop | U-FNO Dir Drop | U-Net+FiLM Int Drop | U-FNO Int Drop |
| --- | ---: | ---: | ---: | ---: |
| **u_wind** | **+20.6pp** | +5.6pp | +0.4pp | +0.4pp |
| **v_wind** | **+10.1pp** | +6.9pp | +1.9pp | +1.7pp |
| **SST** | +0.0pp | +2.1pp | **+4.3pp** | **+5.0pp** |
| geopotential | -0.4pp | -1.1pp | -1.1pp | -1.9pp |
| wind_shear | +0.4pp | +1.1pp | +0.7pp | +1.1pp |
| vorticity | -0.4pp | +1.1pp | +0.4pp | -0.4pp |

This is where the two architectures diverge most clearly:

- **U-Net+FiLM depends heavily on u/v wind for direction** (+20.6pp and +10.1pp drop), far
  more than U-FNO (+5.6pp and +6.9pp). The spatial convolutions are highly specialised for
  reading local wind patterns around the storm centre.
- **Both models rely on SST for intensity** (+4.3pp and +5.0pp). Warm ocean water is the
  primary energy source for cyclone intensification, and both architectures pick this up.
- **U-FNO distributes direction information more evenly** across channels rather than
  concentrating on wind. This may explain why its direction accuracy is lower overall - it
  hasn't specialised as aggressively for the local wind signal.

**Environmental feature group ablation (leave-one-out)**

_Table 12c: Env Feature Group Ablation_

| Feature Group | U-Net+FiLM Dir Drop | U-FNO Dir Drop | U-Net+FiLM Int Drop | U-FNO Int Drop |
| --- | ---: | ---: | ---: | ---: |
| wind | +0.2pp | +0.6pp | -0.4pp | -0.9pp |
| move_velocity | +0.0pp | -0.2pp | -0.2pp | +0.0pp |
| intensity_class | +0.6pp | +0.4pp | -0.6pp | -0.7pp |
| month | -0.6pp | +0.0pp | -0.4pp | +1.7pp |
| history_dir_12h | +0.4pp | +1.1pp | +0.6pp | +0.2pp |
| history_dir_24h | -0.6pp | +0.2pp | +0.7pp | -0.4pp |
| **history_int_24h** | -1.1pp | +0.6pp | **+3.0pp** | +0.7pp |

Individual env features have small effects compared to grid channels, but one stands out:
**history_int_24h** (the 24h intensity change history) costs U-Net+FiLM 3.0pp on intensity
when removed. This makes sense - the U-Net+FiLM relies more on the env vector for
intensity context (as shown in the modality ablation), so losing intensity history hits it
harder. The U-FNO, which extracts intensity signals directly from the grid's spectral
representation, is less affected.

![Modality ablation comparison](figures/ablation_modality_comparison.png)
![Grid channel ablation comparison](figures/ablation_grid_comparison.png)
![Env feature ablation comparison](figures/ablation_env_comparison.png)

### 5.2 SHAP & Gradient Attribution

Ablation tells us *which inputs matter* but not *how* the model uses them internally. To
get a finer-grained view, we apply two attribution methods:

- **SHAP GradientExplainer** [6] on the environmental features: a wrapper class isolates
  the env input by fixing grid, 1D, and time inputs to their dataset means, so SHAP values
  reflect only the marginal contribution of each environmental feature to direction
  predictions.
- **Gradient-based grid attribution**: for each grid channel, we compute the mean absolute
  gradient of the predicted class logit with respect to the input
  (`mean(|d_logit/d_input|)`). This is computationally tractable on the high-dimensional
  grid (15 x 81 x 81) where full SHAP would be prohibitive.

**SHAP on env features**

The SHAP beeswarm plots reveal strikingly different feature importance profiles between
the two models:

- **U-Net+FiLM** assigns the largest SHAP magnitudes to `month` features (especially
  `month_6`, corresponding to peak typhoon season) and `intensity_class`. Its overall SHAP
  magnitudes are ~5x larger than U-FNO's, consistent with the ablation finding that
  U-Net+FiLM depends more on the env vector.
- **U-FNO** assigns its largest SHAP values to `directional history` features
  (`hist_dir_12h`, `hist_dir_24h`), with much smaller overall magnitudes. This confirms
  that the spectral model extracts most of what it needs from the grid and uses env
  features only as a minor supplement.

At the group level, **intensity_class** has the highest mean |SHAP| for U-Net+FiLM
(0.029), roughly 3x the U-FNO value (0.009). This aligns with the ablation result:
U-Net+FiLM relies on the env vector for intensity context that U-FNO captures directly
from the grid's spectral representation.

**Gradient attribution on grid channels**

The per-channel gradient magnitudes confirm the ablation story from a complementary angle:

- **U-Net+FiLM** has its highest gradients on **v_wind and u_wind** channels, with all
  other channels substantially lower. This concentration mirrors the ablation result where
  removing u/v wind caused a 20.6pp/10.1pp direction drop.
- **U-FNO** distributes gradient magnitude more evenly: SST, wind_shear, and vorticity all
  have comparable or higher gradients than in U-Net+FiLM, while wind gradients are lower.
  The spectral convolutions attend to the full grid rather than specialising on local wind
  patterns.

Together, the ablation and attribution results tell a consistent story: **U-Net+FiLM
specialises on local wind fields for direction and relies on env features for intensity
context, while U-FNO extracts both direction and intensity signals from the full grid via
its spectral representation, making it less dependent on any single channel or env
feature.**

![SHAP beeswarm: U-Net+FiLM](figures/shap_env_beeswarm_unet.png)
![SHAP beeswarm: U-FNO](figures/shap_env_beeswarm_ufno.png)
![SHAP env group comparison](figures/shap_env_group_comparison.png)
![Gradient grid channel comparison](figures/gradient_grid_comparison.png)

## Conclusion

This project investigated whether deep learning models trained on the Western Pacific can
generalise to the data-scarce South Pacific for tropical cyclone forecasting.

**Key findings:**

1. **Direction and intensity need different architectures.** U-Net+FiLM is best for
   direction (56.3% WP, 33.8% zero-shot SP) because direction depends on local wind
   patterns that spatial convolutions capture well. U-FNO is best for intensity (60.0% WP,
   42.5% zero-shot SP) because intensity depends on global thermodynamic patterns that
   spectral convolutions capture well. No single model does both tasks well.

2. **Intensity transfers better than direction.** This was our initial hypothesis
   (Introduction), and the results confirm it. The ocean physics that drives intensity
   (SST, wind shear) is shared across hemispheres (Figure 1). Direction, on the other
   hand, is hemisphere-mirrored due to the Coriolis effect, making it fundamentally harder
   to transfer. The ablation study (Part 5) confirms this mechanistically: u/v wind
   channels drive direction, while SST and intensity_class drive intensity.

3. **Temporal conditioning (FiLM) helps intensity in-basin but doesn't improve cross-basin
   transfer.** Adding storm lifecycle and seasonal information gives a +4.3pp intensity
   boost on WP, but this benefit doesn't fully carry over to SP zero-shot evaluation
   (+0.8pp direction recovery after fine-tuning). Temporal patterns may be basin-specific.

4. **Fine-tuning with very little data helps U-Net+FiLM but can hurt U-FNO.** With only
   354 SP samples (~10% of WP training data), U-Net+FiLM recovers modestly (+0.8pp
   direction, +6.3pp intensity), while U-FNO's compact 1.0M parameters are vulnerable to
   catastrophic forgetting (−9.6pp direction). This highlights a trade-off: parameter
   efficiency helps zero-shot transfer but hurts fine-tuning robustness.

5. **The U-FNO's learned gate weights confirm that both global and local features
   matter.** The spectral branch consistently receives the highest weight (~40%), but the
   U-Net branch's weight increases with depth (27% → 31%), suggesting the model first
   captures global context and then refines with local spatial information.

All code, trained checkpoints, and analysis notebooks are available in this repository.

## References

[1] Huang, C., Mu, P., Zhang, J., et al. (2025) 'Benchmark dataset and deep learning
method for global tropical cyclone forecasting', *Nature Communications*, 16, 5923.

[2] DeMaria, M. and Kaplan, J. (1994) 'A Statistical Hurricane Intensity Prediction Scheme
(SHIPS) for the Atlantic Basin', *Weather and Forecasting*, 9(2), pp. 209-220.

[3] Perez, E., Strub, F., de Vries, H., Dumoulin, V. and Courville, A. (2018) 'FiLM:
Visual Reasoning with a General Conditioning Layer', *Proceedings of the AAAI Conference
on Artificial Intelligence*.

[4] Li, Z., Kovachki, N., Azizzadenesheli, K., Liu, B., Bhattacharya, K., Stuart, A. and
Anandkumar, A. (2021) 'Fourier Neural Operator for Parametric Partial Differential
Equations', *International Conference on Learning Representations (ICLR)*.

[5] Wen, G., Li, Z., Azizzadenesheli, K., Anandkumar, A. and Benson, S.M. (2022) 'U-FNO:
An enhanced Fourier neural operator-based deep-learning model for multiphase flow',
*Advances in Water Resources*, 163, 104180.

[6] Lundberg, S.M. and Lee, S.-I. (2017) 'A Unified Approach to Interpreting Model
Predictions', *Advances in Neural Information Processing Systems (NeurIPS)*, pp.
4766-4777.

[7] Qu, H., et al. (2025) 'Accurate tropical cyclone intensity forecasts using a non-
iterative spatiotemporal transformer model', *npj Climate and Atmospheric Science*.

[8] Pathak, J., Subramanian, S., Harrington, P., et al. (2022) 'FourCastNet: A Global
Data-driven High-resolution Weather Model using Adaptive Fourier Neural Operators', *arXiv
preprint*, arXiv:2202.11214.